In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
movies_metadata_df = spark.read.format("delta").load(f"{silver_folder_path}/movies_metadata")
ratings_df = spark.read.format("delta").load(f"{silver_folder_path}/ratings")
links_df = spark.read.format("delta").load(f"{silver_folder_path}/links")
crew_df = spark.read.format("delta").load(f"{silver_folder_path}/crew")

movies_metadata_df.printSchema()
ratings_df.printSchema()
links_df.printSchema()
crew_df.printSchema()


In [0]:
from pyspark.sql import functions as F

final_movies_df = (
    ratings_df.join(links_df, links_df.movie_id == ratings_df.movie_id, "inner")
    .join(movies_metadata_df, links_df.tmbd_id == movies_metadata_df.id, "inner")
    .groupBy(movies_metadata_df.id, "title", "budget", "revenue", "collection_id", "collection_name")
    .agg(
        F.avg("rating").alias("average_rating"),
        F.count("user_id").alias("number_of_ratings"),
    )
    .filter("number_of_ratings > 10")
    .filter("collection_id is not null")
    .withColumn("monetary_performance", F.expr("try_divide(revenue, budget)"))
    .orderBy(F.col("collection_id"), F.col("average_rating").desc())
)
display(final_movies_df)

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy("collection_id")

df = (
  final_movies_df.withColumn("collection_rating", F.avg("average_rating").over(w))
  .orderBy(F.col("collection_rating").desc(), F.col("average_rating").desc())
  .select("title", "collection_name", "average_rating")
)
display(df)

In [0]:
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Collection average on every row (same idea as Bond actor avg)
w = Window.partitionBy("collection_name")
df_plot = (
  df
    .withColumn("collection_avg", F.avg("average_rating").over(w))
    .withColumn("films_in_collection", F.count("*").over(w))
    .filter(F.col("films_in_collection") >= 2)  # skip one-film "collections"
)

# --- Chart 1: top collections by average rating ---
top_collections_pdf = (
  df_plot
    .select("collection_name", "collection_avg", "films_in_collection")
    .dropDuplicates(["collection_name"])
    .orderBy(F.col("collection_avg").desc())
    .limit(15)
    .toPandas()
)

fig1 = px.bar(
  top_collections_pdf,
  x="collection_avg",
  y="collection_name",
  orientation="h",
  hover_data=["films_in_collection"],
  text=top_collections_pdf["collection_avg"].round(2),
  title="Top 15 collections by average MovieLens rating (≥2 films)",
  labels={
    "collection_avg": "Collection average rating",
    "collection_name": "Collection",
  },
  range_x=[3.5, 4.5],
)
fig1.update_traces(textposition="outside", cliponaxis=False)
fig1.update_layout(yaxis={"categoryorder": "total ascending"}, height=550, margin=dict(r=80))
fig1.show()

# --- Chart 2: films in those top collections vs